In [ ]:
#!/usr/bin/env python3
"""
pcap_filter.py
==============
Batch-filters all .pcap / .pcapng files in an input folder,
keeping only TCP and UDP packets, and writes filtered copies
to an output folder.

Filtered out:
    - ARP / raw Ethernet (no IP layer)
    - ICMPv6 Neighbour/Router Discovery (LAN noise)
    - IGMP, multicast
    - Any other non-TCP/UDP protocol

Kept:
    - TCP  (all: handshake, TLS, data, FIN)
    - UDP  (all: DNS port 53, QUIC port 443, and everything else UDP)

Usage:
    python pcap_filter.py <input_folder> [output_folder]

    If output_folder is omitted, a sibling folder named
    "<input_folder>_filtered" is created automatically.

Dependencies:
    pip install scapy
"""

import sys
import argparse
from pathlib import Path

try:
    from scapy.all import (
        rdpcap, wrpcap,
        IP, IPv6, TCP, UDP,
        ICMPv6ND_RA, ICMPv6ND_RS,
        ICMPv6ND_NS, ICMPv6ND_NA,
        ICMPv6MLReport, ICMPv6MLQuery,
    )
except ImportError:
    sys.exit("scapy is not installed.  Run:  pip install scapy")


# ── noise types to always drop ────────────────────────────────────────────────

_ICMPV6_NOISE = (
    ICMPv6ND_RA,        # Router Advertisement    (your Arcadyan router)
    ICMPv6ND_RS,        # Router Solicitation
    ICMPv6ND_NS,        # Neighbour Solicitation
    ICMPv6ND_NA,        # Neighbour Advertisement
    ICMPv6MLReport,     # Multicast Listener Report
    ICMPv6MLQuery,      # Multicast Listener Query
)


# ── per-packet decision ───────────────────────────────────────────────────────

def keep(pkt) -> bool:
    """
    Return True if the packet should be kept.

    Rules (in order):
      1. Must have an IP or IPv6 layer — drops ARP, raw Ethernet broadcasts
      2. Drop ICMPv6 LAN noise (NDP, multicast)
      3. Keep if TCP layer present
      4. Keep if UDP layer present
      5. Drop everything else (ICMP echo, OSPF, etc.)
    """
    # rule 1 — must be an IP packet
    if not (pkt.haslayer(IP) or pkt.haslayer(IPv6)):
        return False

    # rule 2 — drop ICMPv6 LAN noise
    if any(pkt.haslayer(t) for t in _ICMPV6_NOISE):
        return False

    # rule 3 — keep TCP (covers TLS, HTTP, everything over TCP)
    if pkt.haslayer(TCP):
        return True

    # rule 4 — keep UDP (covers DNS, QUIC, NTP, etc.)
    if pkt.haslayer(UDP):
        return True

    # rule 5 — drop (ICMP, IGMP, raw IPv6, etc.)
    return False


# ── per-file processing ───────────────────────────────────────────────────────

def filter_pcap(src: Path, dst: Path) -> dict:
    """
    Read src, filter, write to dst.
    Returns a stats dict.
    """
    try:
        packets = rdpcap(str(src))
    except Exception as e:
        return {"file": src.name, "error": str(e), "total": 0, "kept": 0}

    total    = len(packets)
    filtered = [p for p in packets if keep(p)]
    kept     = len(filtered)
    dropped  = total - kept

    if filtered:
        wrpcap(str(dst), filtered)
    else:
        # write an empty-but-valid pcap rather than nothing
        wrpcap(str(dst), [])

    return {
        "file":    src.name,
        "total":   total,
        "kept":    kept,
        "dropped": dropped,
        "pct":     f"{100 * kept / total:.1f}%" if total else "n/a",
        "error":   None,
    }


# ── main ──────────────────────────────────────────────────────────────────────

def main():
    in_dir  = Path(r"E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_captures")
    out_dir = in_dir

    if not in_dir.exists():
        sys.exit(f"Input folder not found: {in_dir}")

    pcap_files = sorted(
        list(in_dir.glob("*.pcap")) + list(in_dir.glob("*.pcapng"))
    )

    if not pcap_files:
        sys.exit(f"No .pcap / .pcapng files found in: {in_dir}")

    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nInput  : {in_dir}")
    print(f"Output : {out_dir}")
    print(f"Files  : {len(pcap_files)}\n")
    print(f"{'File':<60} {'Total':>7} {'Kept':>7} {'Dropped':>9} {'%Kept':>7}")
    print("─" * 92)

    total_in = total_out = 0

    for src in pcap_files:
        dst  = out_dir / (src.stem + "_filtered" + src.suffix)
        stat = filter_pcap(src, dst)

        if stat["error"]:
            print(f"  {'ERROR':6}  {src.name}  →  {stat['error']}")
            continue

        total_in  += stat["total"]
        total_out += stat["kept"]

        print(
            f"{stat['file']:<60} "
            f"{stat['total']:>7,} "
            f"{stat['kept']:>7,} "
            f"{stat['dropped']:>9,} "
            f"{stat['pct']:>7}"
        )

    print("─" * 92)
    dropped_total = total_in - total_out
    pct = f"{100 * total_out / total_in:.1f}%" if total_in else "n/a"
    print(
        f"{'TOTAL':<60} "
        f"{total_in:>7,} "
        f"{total_out:>7,} "
        f"{dropped_total:>9,} "
        f"{pct:>7}"
    )
    print(f"\nDone. Filtered files saved to: {out_dir}\n")

if __name__ == "__main__":
    main()


Input  : E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_captures
Output : E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\processed_datasets\filtered_pcaps
Files  : 129

File                                                           Total    Kept   Dropped   %Kept
────────────────────────────────────────────────────────────────────────────────────────────


20260527T112337Z_groq_qwen3_32b_short_factual_504f162c.pcap      609     583        26   95.7%
20260527T112345Z_groq_qwen3_32b_short_factual_e4c6b426.pcap      418     404        14   96.7%
20260527T112352Z_groq_qwen3_32b_short_factual_c0e8478a.pcap      621     599        22   96.5%
20260527T112400Z_groq_qwen3_32b_short_factual_0047b45f.pcap      942     920        22   97.7%
20260527T112410Z_groq_qwen3_32b_short_factual_2d06157a.pcap      279     264        15   94.6%
20260527T112417Z_groq_qwen3_32b_short_factual_4e769e39.pcap    1,659   1,626        33   98.0%
20260527T112428Z_groq_qwen3_32b_short_factual_f3d8ad3f.pcap    1,543   1,509        34   97.8%
20260527T112440Z_groq_qwen3_32b_short_factual_703c41d9.pcap      472     460        12   97.5%
20260527T112448Z_groq_qwen3_32b_short_factual_7a2ddd66.pcap      912     897        15   98.4%
20260527T112456Z_groq_qwen3_32b_long_generation_e9bdfbd6.pcap   2,038   1,993        45   97.8%
20260527T112509Z_groq_qwen3_32b_long_generation_f